# GlycoShape ReGlyco Workflow Demo (Colab)



This notebook demonstrates the **exact API workflow** used by the GlycoShape webapp, extended with **protein binder generation**:



1. Upload a **target protein PDB** → create a session

2. Run **GlcNAc scanning** (ReGlyco `scan`) → obtain passed attachment sites

3. Run **ReGlycoEnsemble** with **SASA + hotspots** on all passed sites using **G00028MO**

4. Visualize the produced `hotspots.pdb` with **Mol**

5. Use **RFD3** to generate a **binder–target complex** conditioned on hotspots

6. Re-scan the complex and run **two ReGlyco optimization jobs** (rotamer **off** and **on**) using **G00028MO**



API base: `https://glycoshape.org`


In [ ]:
#@title Colab setup (GPU required) { display-mode: "form" }


# This notebook can run as-is in Google Colab (GPU runtime required for RFD3).


import os


from pathlib import Path




os.environ.setdefault('CCD_MIRROR_PATH', '')


os.environ.setdefault('PDB_MIRROR_PATH', '')




READY = Path('FOUNDRY_READY')


if not READY.exists():


    print('Installing rc-foundry (includes AtomWorks + RFD3 runtime)...')


    # Avoid common operator conflicts in Colab


    os.system('pip uninstall -y torchvision >/dev/null 2>&1')


    os.system("pip install -q 'rc-foundry[all]'")


    # Download weights (RFD3 only for this notebook)


    os.system('foundry install rfd3')


    READY.write_text('ok\n')


    print('Done.')


else:


    print('rc-foundry already installed (FOUNDRY_READY present).')




# Quick CUDA sanity check


try:


    import torch


    if torch.cuda.is_available():


        print(f'CUDA available: {torch.cuda.device_count()} GPU(s)')


    else:


        print('CUDA not available (binder generation will be slow/unsupported).')


except Exception:


    pass

Installing rc-foundry (includes AtomWorks + RFD3 runtime)...
Done.
CUDA available: 1 GPU(s)


In [ ]:
#@title Setup (run once) { display-mode: "form" }


import time


from pathlib import Path


from typing import Any, Dict, List, Optional, Tuple




import requests




API_BASE = 'https://glycoshape.org'




def _raise_for_api(resp: requests.Response) -> None:


    if resp.ok:


        return


    try:


        payload = resp.json()


        msg = payload.get('error') or payload


    except Exception:


        msg = resp.text


    raise RuntimeError(f'API error {resp.status_code}: {msg}')




def create_session_from_pdb_bytes(filename: str, pdb_bytes: bytes, *, timeout_s: int = 600) -> Dict[str, Any]:


    files = {'protFile': (filename, pdb_bytes, 'chemical/x-pdb')}


    resp = requests.post(f'{API_BASE}/api/sessions', files=files, timeout=timeout_s)


    _raise_for_api(resp)


    return resp.json()




def submit_job(


    session_uuid: str,


    job_type: str,


    *,


    selected_glycans: Optional[Dict[str, str]] = None,


    parameters: Optional[Dict[str, Any]] = None,


    timeout_s: int = 60,


) -> Dict[str, Any]:


    payload: Dict[str, Any] = {


        'session_uuid': session_uuid,


        'jobType': job_type,


    }


    if selected_glycans is not None:


        payload['selectedGlycans'] = selected_glycans


    if parameters is not None:


        payload['parameters'] = parameters




    resp = requests.post(f'{API_BASE}/api/jobs', json=payload, timeout=timeout_s)


    _raise_for_api(resp)


    return resp.json()




def get_job(job_uuid: str, *, timeout_s: int = 30) -> Dict[str, Any]:


    resp = requests.get(f'{API_BASE}/api/jobs/{job_uuid}', timeout=timeout_s)


    _raise_for_api(resp)


    return resp.json()




def wait_job(job_uuid: str, *, poll_s: float = 2.0, timeout_s: int = 3600) -> Dict[str, Any]:


    t0 = time.time()


    last_status = None


    while True:


        meta = get_job(job_uuid)


        status = (meta.get('status')


                  or (meta.get('queue') or {}).get('status')


                  or ((meta.get('results') or {}).get('status')))


        if status != last_status:


            print(f'[{job_uuid[:8]}] status: {status}')


            last_status = status




        if status in {'completed', 'finished', 'failed'}:


            if status == 'failed':


                err = (meta.get('results') or {}).get('error') or meta.get('error')


                raise RuntimeError(f'Job failed: {job_uuid} ({err})')


            return meta




        if time.time() - t0 > timeout_s:


            raise TimeoutError(f'Timed out waiting for job: {job_uuid}')




        time.sleep(poll_s)




def job_file_url(job_uuid: str, filename: str) -> str:


    return f'{API_BASE}/api/jobs/{job_uuid}/files/{filename}'




def scan_passed_sites(job_meta: Dict[str, Any]) -> List[str]:


    items = ((job_meta.get('results') or {}).get('results')) or []


    passed = [it.get('residue') for it in items if it.get('clash_solved') is True and it.get('residue')]


    # Preserve order, drop duplicates


    seen = set()


    out: List[str] = []


    for r in passed:


        if r not in seen:


            seen.add(r)


            out.append(r)


    return out




def load_pdb_bytes(path_or_name: str) -> Tuple[str, bytes]:


    p = Path(path_or_name)


    candidates = [


        p,


        Path('Colab_demo') / path_or_name,


        Path('Colab_demo') / p.name,


        Path(p.name),


    ]


    for c in candidates:


        if c.exists() and c.is_file():


            return c.name, c.read_bytes()


    raise FileNotFoundError(f'PDB not found: {path_or_name} (tried: ' + ', '.join(str(x) for x in candidates) + ')')




def colab_upload_one() -> Tuple[str, bytes]:


    from google.colab import files  # type: ignore


    uploaded = files.upload()


    name = next(iter(uploaded.keys()))


    return name, uploaded[name]




def get_pdb_input(default_filename: str) -> Tuple[str, bytes]:


    """Colab: upload dialog (unless file exists already). Local/VS Code: reads a local file path."""


    # If the file already exists (e.g., generated by an earlier cell), always reuse it.


    try:


        name, b = load_pdb_bytes(default_filename)


        return name, b


    except Exception:


        pass




    try:


        import google.colab  # type: ignore


        _ = google.colab  # silence unused


        return colab_upload_one()


    except Exception:


        return load_pdb_bytes(default_filename)

## 1) Target protein: upload PDB → session
Upload a target protein PDB file (single structure).

In [ ]:
#@title Target protein PDB → session { display-mode: "form" }

TARGET_PDB_PATH = 'target.pdb'  # or a full path

target_name, target_bytes = get_pdb_input(TARGET_PDB_PATH)
print('Loaded:', target_name, f'({len(target_bytes):,} bytes)')

target_session = create_session_from_pdb_bytes(target_name, target_bytes)
target_session_uuid = target_session['session_uuid']
print('session_uuid:', target_session_uuid)


Saving AF-P01588-F1-model_v6.pdb to AF-P01588-F1-model_v6.pdb
Loaded: AF-P01588-F1-model_v6.pdb (126,521 bytes)
session_uuid: 37ff0353-0964-4d8a-9e8a-c39667943882


## 2) GlcNAc scanning (ReGlyco `scan`)
This finds candidate glycosylation locations and attempts a GlcNAc attachment scan.

We will use **passed** sites (`clash_solved == True`) for downstream ensemble.

In [ ]:
scan_job = submit_job(target_session_uuid, 'scan')
scan_job_uuid = scan_job['job_uuid']
scan_meta = wait_job(scan_job_uuid)

passed_sites = scan_passed_sites(scan_meta)
print('Passed sites:', len(passed_sites))
passed_sites

[c11c11f1] status: completed
Passed sites: 3


['51_A', '65_A', '110_A']

## 3) ReGlycoEnsemble on passed sites (G00028MO) + hotspots
This runs `ensemble` on all scan-passed residues using **G00028MO**, with **SASA + hotspots** enabled.

Hotspot residues will be encoded as **B-factor = 100** in `hotspots.pdb`.

In [ ]:
# @title
GLYCAN_ID = 'G00028MO'
selected = {site: GLYCAN_ID for site in passed_sites}

ensemble_params = {
    # Keep these modest for a fast demo; increase for production.
    'ensembleSize': 50,
    'calculateSASA': True,
    'calculateHotspots': True,
    'outputFormat': 'PDB',
}

ensemble_job = submit_job(target_session_uuid, 'ensemble', selected_glycans=selected, parameters=ensemble_params)
ensemble_job_uuid = ensemble_job['job_uuid']
_ = wait_job(ensemble_job_uuid)

hotspots_pdb_url = job_file_url(ensemble_job_uuid, 'hotspots.pdb')
print('hotspots.pdb:', hotspots_pdb_url)

[f6da0f79] status: queued
[f6da0f79] status: leased
[f6da0f79] status: completed
hotspots.pdb: https://glycoshape.org/api/jobs/f6da0f79-f54a-4476-9e16-e2f9dc91f8ab/files/hotspots.pdb


In [ ]:
# @title
# Parse hotspot residues (B-factor = 100)

hotspots_pdb = requests.get(hotspots_pdb_url, timeout=60).text.splitlines()

hotspot_sites = set()
for line in hotspots_pdb:
    if not (line.startswith('ATOM') or line.startswith('HETATM')):
        continue
    # PDB columns: chainID(22), resSeq(23-26), B-factor(61-66)
    chain = (line[21] or '?').strip() or '?'
    resi = line[22:26].strip()
    try:
        b = float(line[60:66])
    except Exception:
        continue
    if b >= 99.9 and resi:
        hotspot_sites.add(f"{resi}_{chain}")

hotspot_sites = sorted(hotspot_sites, key=lambda s: (int(s.split('_')[0]), s.split('_')[1]))
print(f"Hotspot residues (B=100): {len(hotspot_sites)}")


Hotspot residues (B=100): 167


## 4) Visualization (Mol)
This cell loads the resulting `hotspots.pdb` directly from the GlycoShape API.

In [ ]:
# @title
from IPython.display import HTML

pdb_url = hotspots_pdb_url

HTML(f"""
<div id="molstar" style="width:100%; height:600px; position:relative;"></div>

<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.css" />
<script src="https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.js"></script>

<script>
async function startMolstar() {{
    const viewer = await molstar.Viewer.create('molstar', {{
        layoutIsExpanded: false,
        layoutShowControls: false,
        layoutShowRemoteState: false
    }});

    const plugin = viewer.plugin;

    // 1. Download the data explicitly
    const data = await plugin.builders.data.download({{ url: "{pdb_url}" }});

    // 2. Parse the trajectory and create the model
    const trajectory = await plugin.builders.structure.parseTrajectory(data, 'pdb');
    const model = await plugin.builders.structure.createModel(trajectory);
    const structure = await plugin.builders.structure.createStructure(model);

    // 3. Create a component targeting 'all' atoms
    const allAtoms = await plugin.builders.structure.tryCreateComponentStatic(structure, 'all');

    if (allAtoms) {{
        // 4. Add the original cartoon representation
        await plugin.builders.structure.representation.addRepresentation(allAtoms, {{
            type: 'cartoon',
            color: 'uncertainty'
        }});

        // 5. Add the new semi-transparent Gaussian surface
        await plugin.builders.structure.representation.addRepresentation(allAtoms, {{
            type: 'gaussian-surface',
            typeParams: {{ alpha: 0.15 }}, // This controls the opacity
            color: 'uncertainty'
        }});
    }}
}}

startMolstar();
</script>
""")

## 5) Binder generation (RFD3): hotspots → complex.pdb

This step uses **RFdiffusion3 (RFD3)** to generate a *protein binder* conditioned on the GlycoShape **hotspot residues**.

**Output:** generated complexes saved under `binder_design_out/` as `complex_generated_00001.pdb`, `complex_generated_00002.pdb`, ...

Notes:

- Default `n_binders=3` is chosen for **free Colab** runtimes; increase for production.

- If no hotspots are returned, we fall back to auto-selecting residues along the target chain.


In [ ]:
#@title RFD3 binder generation (hotspot-conditioned) { display-mode: "form" }

import json

from pathlib import Path

import numpy as np


# ---- User-adjustable parameters ----

# Number of binder designs to generate
N_BINDERS = 10  #@param {type:"slider", min:1, max:10, step:1}

BINDER_LENGTH = 60

BINDER_LENGTH_WINDOW = 10     # generates [L-window, L+window]

N_HOTSPOTS_FOR_RFD3 = 5       # small conditioning set is usually sufficient

# If True, vary the hotspot subset per binder to encourage diversity
DIVERSIFY_HOTSPOTS = True  #@param {type:"boolean"}
HOTSPOT_DIVERSITY_MODE = 'rotate'  #@param ["rotate", "random"]

DIFFUSION_BATCH_SIZE = 1

SEED = 0

STEP_SCALE = 3.0

GAMMA_0 = 0.2


OUT_DIR = Path('binder_design_out').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---- Parse target PDB from the earlier upload ----
# (We reuse `target_bytes` from step 1 and write it to disk for parsers.)

target_pdb_path = (OUT_DIR / 'target_input.pdb')
target_pdb_path.write_bytes(target_bytes)


import biotite.structure as bs
from biotite.structure.io.pdb import PDBFile
from biotite.structure.io.pdb import get_structure as get_pdb_structure


pdb = PDBFile.read(str(target_pdb_path))
aa_all = get_pdb_structure(pdb, model=1)


def _infer_target_chain(hotspot_sites, passed_sites, atom_array) -> str:
    # Prefer the chain seen in GlycoShape hotspots, else scan-passed sites, else first chain in the PDB.

    def _chain_from_sites(sites):
        chains = []
        for s in sites:
            try:
                _, c = s.split('_', 1)
                if c:
                    chains.append(c)
            except Exception:
                pass
        if not chains:
            return None
        # most frequent
        return sorted(set(chains), key=lambda x: (-chains.count(x), x))[0]

    c = _chain_from_sites(hotspot_sites)
    if c:
        return c

    c = _chain_from_sites(passed_sites)
    if c:
        return c

    chains = sorted(set(atom_array.chain_id.tolist()))
    return chains[0] if chains else 'A'


TARGET_CHAIN = _infer_target_chain(hotspot_sites if 'hotspot_sites' in globals() else [], passed_sites, aa_all)


# Filter to the target chain and amino acids only
chain_mask = (aa_all.chain_id == TARGET_CHAIN)
aa_mask = bs.filter_amino_acids(aa_all)
target_chain_atoms = aa_all[chain_mask & aa_mask]


if len(target_chain_atoms) == 0:
    raise ValueError(f'No amino-acid atoms found for chain {TARGET_CHAIN!r} in target PDB')


# Renumber residues sequentially: old res_id -> 1..N (RFD3 contig/hotspots assume contiguous numbering)
old_res_ids = np.sort(np.unique(target_chain_atoms.res_id))
resid_map = {int(old): int(i+1) for i, old in enumerate(old_res_ids)}

target_chain_atoms = target_chain_atoms.copy()
target_chain_atoms.res_id = np.array([resid_map[int(r)] for r in target_chain_atoms.res_id], dtype=target_chain_atoms.res_id.dtype)


target_seq_len = int(len(np.unique(target_chain_atoms.res_id)))
print(f'Target chain: {TARGET_CHAIN} ({target_seq_len} residues after renumbering)')


# Write normalized target CIF for RFD3 input
from atomworks.io.utils.io_utils import to_cif_file


normalized_target_cif = (OUT_DIR / 'target_normalized.cif')
to_cif_file(target_chain_atoms, str(normalized_target_cif))
print('Wrote:', normalized_target_cif)


# ---- Build select_hotspots from GlycoShape hotspots (B-factor=100) ----

def _parse_hotspots_for_chain(hotspot_sites, chain: str):
    out = []
    for s in hotspot_sites:
        try:
            resi, c = s.split('_', 1)
            if c == chain:
                out.append(int(resi))
        except Exception:
            continue
    out = sorted(set(out))
    return out


raw_hotspot_resids = _parse_hotspots_for_chain(hotspot_sites if 'hotspot_sites' in globals() else [], TARGET_CHAIN)


def _evenly_spaced(items, k):
    if not items:
        return []
    k = int(max(1, min(int(k), len(items))))
    idx = np.linspace(0, len(items) - 1, k, dtype=int)
    return [items[i] for i in idx]


# Map original residue ids -> renumbered residue ids; drop any that are missing
mapped_hotspots = []
for old_res in raw_hotspot_resids:
    if int(old_res) in resid_map:
        mapped_hotspots.append(resid_map[int(old_res)])

mapped_hotspots = sorted(set(mapped_hotspots))


if mapped_hotspots:
    chosen_hotspots = _evenly_spaced(mapped_hotspots, N_HOTSPOTS_FOR_RFD3)
    source = 'GlycoShape hotspots.pdb (mapped to renumbered target)'
else:
    # Fallback: auto-pick evenly spaced residues across the chain
    all_res = list(range(1, target_seq_len + 1))
    chosen_hotspots = _evenly_spaced(all_res, N_HOTSPOTS_FOR_RFD3)
    source = 'fallback (evenly spaced along target chain)'

# Pool used for diversification: prefer mapped hotspots if available, else whole chain
hotspot_pool = list(mapped_hotspots) if mapped_hotspots else list(range(1, target_seq_len + 1))

def _pick_hotspots(pool: list[int], k: int, *, binder_idx: int, mode: str, seed: int) -> list[int]:
    if not pool:
        return []
    pool = sorted(set(int(x) for x in pool))
    k = int(max(1, min(int(k), len(pool))))

    mode = (mode or '').strip().lower()
    if mode == 'random':
        rng = np.random.default_rng(int(seed) + int(binder_idx))
        # Sample without replacement for this binder
        return sorted(rng.choice(pool, size=k, replace=False).tolist())

    # rotate (deterministic): take k consecutive elements with an offset
    # offset advances with binder_idx so each binder sees a shifted subset
    step = max(1, len(pool) // max(1, int(N_BINDERS)))
    offset = (int(binder_idx) * step) % len(pool)
    doubled = pool + pool
    picked = doubled[offset: offset + k]
    return sorted(set(int(x) for x in picked))


def _atom_for_res(res_id: int) -> str:
    m = (target_chain_atoms.res_id == res_id)
    if not np.any(m):
        return 'CB'
    res_name = str(target_chain_atoms.res_name[m][0])
    return 'CA' if res_name.upper() == 'GLY' else 'CB'


select_hotspots = {f"{TARGET_CHAIN}{int(r)}": _atom_for_res(int(r)) for r in chosen_hotspots}
print('Hotspot source:', source)
print('select_hotspots (single):', select_hotspots)


# ---- RFD3 binder design input spec ----

binder_min = max(1, int(BINDER_LENGTH) - int(BINDER_LENGTH_WINDOW))
binder_max = int(BINDER_LENGTH) + int(BINDER_LENGTH_WINDOW)


base_binder_spec = {
    'dialect': 2,
    'infer_ori_strategy': 'hotspots',
    'input': str(normalized_target_cif),
    'contig': f"{binder_min}-{binder_max},/0,{TARGET_CHAIN}1-{target_seq_len}",
    'is_non_loopy': True,
}

if bool(DIVERSIFY_HOTSPOTS):
    # One input per binder, each with its own select_hotspots
    rfd3_input_spec = {}
    per_binder_select_hotspots = []
    for i in range(1, int(N_BINDERS) + 1):
        hs = _pick_hotspots(hotspot_pool, int(N_HOTSPOTS_FOR_RFD3), binder_idx=i, mode=str(HOTSPOT_DIVERSITY_MODE), seed=int(SEED))
        sel = {f"{TARGET_CHAIN}{int(r)}": _atom_for_res(int(r)) for r in hs}
        per_binder_select_hotspots.append(sel)
        rfd3_input_spec[f'binder_design_{i:05d}'] = {**base_binder_spec, 'select_hotspots': sel}
    print('Per-binder select_hotspots (diversified):')
    for i, sel in enumerate(per_binder_select_hotspots, start=1):
        print(f'  Binder {i}:', sel)
else:
    # Single input; RFD3 will generate multiple designs for the same hotspot set
    rfd3_input_spec = {'binder_design': {**base_binder_spec, 'select_hotspots': select_hotspots}}


input_json_path = (OUT_DIR / 'rfd3_binder_input.json')
input_json_path.write_text(json.dumps(rfd3_input_spec, indent=2) + '\n')
print('Wrote:', input_json_path)


# ---- Run RFD3 ----

from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine


effective_batch_size = 1 if bool(DIVERSIFY_HOTSPOTS) else int(DIFFUSION_BATCH_SIZE)
config = RFD3InferenceConfig(diffusion_batch_size=int(effective_batch_size), seed=int(SEED), verbose=True)
try:
    config.inference_sampler['step_scale'] = float(STEP_SCALE)
    config.inference_sampler['gamma_0'] = float(GAMMA_0)
except Exception:
    pass

engine = RFD3InferenceEngine(**config.__dict__)

# RFD3 runs by batches; in diversified mode we request 1 batch (one output per input)
n_batches = 1 if bool(DIVERSIFY_HOTSPOTS) else int(np.ceil(int(N_BINDERS) / int(max(1, effective_batch_size))))
outputs = engine.run(inputs=str(input_json_path), out_dir=None, n_batches=n_batches)


def _collect_rfd3_outputs(outputs_dict):
    """Return list of (key, index, result) in stable key order."""
    out = []
    if not isinstance(outputs_dict, dict):
        return out
    for k in sorted(outputs_dict.keys()):
        v = outputs_dict[k]
        vs = list(v) if isinstance(v, (list, tuple)) else [v]
        vs = [x for x in vs if hasattr(x, 'atom_array')]
        for j, x in enumerate(vs):
            out.append((str(k), int(j), x))
    return out


named_results = _collect_rfd3_outputs(outputs)
if bool(DIVERSIFY_HOTSPOTS):
    # Expect at least one output per binder-design key; take the first from each key
    flat_results = []
    by_key = {}
    for k, j, x in named_results:
        by_key.setdefault(k, []).append((j, x))
    for k in sorted(by_key.keys()):
        xs = [x for _, x in sorted(by_key[k], key=lambda t: t[0])]
        if xs:
            flat_results.append(xs[0])
else:
    flat_results = [x for _, _, x in named_results]
if not flat_results:
    raise RuntimeError('RFD3 did not return any results with atom_array')

# Keep only the requested number of binders
flat_results = flat_results[: int(N_BINDERS)]
if len(flat_results) < int(N_BINDERS):
    print(f'Warning: requested N_BINDERS={int(N_BINDERS)}, but only got {len(flat_results)} result(s).')


# Save complexes for each binder
generated_complex_pdb_paths = []

def _normalize_complex_target_chain_ids(atom_array: bs.AtomArray, *, target_len: int, desired_chain: str = 'A') -> tuple[bs.AtomArray, dict[str, str]]:
    """Rename chains so the *target* protein is always `desired_chain` (default: 'A')."""

    # Work on amino acids only to avoid glycans/ligands skewing counts
    try:
        aa = atom_array[bs.filter_amino_acids(atom_array)]
    except Exception:
        aa = atom_array

    chain_ids = sorted(set(getattr(aa, 'chain_id', []).tolist())) if len(aa) else []
    if not chain_ids:
        return atom_array, {}

    # Count unique residues per chain
    chain_counts: dict[str, int] = {}
    for c in chain_ids:
        try:
            m = (aa.chain_id == c)
            chain_counts[c] = int(len(np.unique(aa.res_id[m])))
        except Exception:
            chain_counts[c] = 0

    # Identify the target chain as the one whose residue count best matches target_len
    def _rank(c: str):
        n = chain_counts.get(c, 0)
        return (abs(int(n) - int(target_len)), -int(n), str(c))

    target_old = sorted(chain_ids, key=_rank)[0]

    # Assign new chain IDs: target -> desired_chain; others -> B, C, ... (skipping desired_chain)
    letters = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
    pool = [x for x in letters if x != desired_chain]
    mapping: dict[str, str] = {str(target_old): str(desired_chain)}
    j = 0
    for c in chain_ids:
        if str(c) == str(target_old):
            continue
        if j >= len(pool):
            # Extremely unlikely for protein complexes here; keep original if we run out
            mapping[str(c)] = str(c)
            continue
        mapping[str(c)] = pool[j]
        j += 1

    out = atom_array.copy()
    # Apply mapping to *all* atoms (including glycans/ligands), so chain IDs stay consistent
    out.chain_id = np.array([mapping.get(str(c), str(c)) for c in out.chain_id], dtype=out.chain_id.dtype)
    return out, mapping

for i, r in enumerate(flat_results, start=1):
    complex_atoms = r.atom_array
    # Normalize chain IDs so target is chain 'A' (binder becomes 'B', etc.)
    try:
        complex_atoms, chain_map = _normalize_complex_target_chain_ids(complex_atoms, target_len=target_seq_len, desired_chain='A')
        if chain_map:
            print(f'[{i}] Chain renaming:', chain_map)
    except Exception as e:
        print(f'[{i}] Chain renaming skipped due to error:', e)

    # Archive CIF
    complex_cif_path = (OUT_DIR / f'complex_{i:05d}.cif')
    to_cif_file(complex_atoms, str(complex_cif_path))

    # PDB for ReGlyco API upload
    complex_pdb_path = (OUT_DIR / f'complex_generated_{i:05d}.pdb')
    pdb_out = PDBFile()
    pdb_out.set_structure(complex_atoms)
    pdb_out.write(str(complex_pdb_path))

    generated_complex_pdb_paths.append(str(complex_pdb_path))

    print(f'[{i}/{len(flat_results)}] Saved complex CIF:', complex_cif_path)
    print(f'[{i}/{len(flat_results)}] Saved complex PDB:', complex_pdb_path)

# Backwards-compat: also write the first binder to the old filename
try:
    if generated_complex_pdb_paths:
        legacy_path = Path('complex_generated.pdb').resolve()
        legacy_path.write_bytes(Path(generated_complex_pdb_paths[0]).read_bytes())
        print('Also wrote legacy:', legacy_path)
except Exception:
    pass

print('Generated binders:', len(generated_complex_pdb_paths))
print('generated_complex_pdb_paths[0:5]:', generated_complex_pdb_paths[:5])


INFO:foundry.inference_engines.base:[rank: 0] Using checkpoint from default installation directory, got: /root/.foundry/checkpoints/rfd3_latest.ckpt
INFO:foundry.inference_engines.base:[rank: 0] Seeding everything with seed=0...
INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0


Target chain: A (193 residues after renumbering)
Wrote: /content/binder_design_out/target_normalized.cif
Hotspot source: GlycoShape hotspots.pdb (mapped to renumbered target)
select_hotspots (single): {'A1': 'CB', 'A42': 'CB', 'A92': 'CB', 'A148': 'CB', 'A193': 'CB'}
Per-binder select_hotspots (diversified):
  Binder 1: {'A17': 'CB', 'A18': 'CB', 'A19': 'CB', 'A20': 'CB', 'A21': 'CB'}
  Binder 2: {'A33': 'CB', 'A34': 'CB', 'A35': 'CB', 'A36': 'CB', 'A37': 'CB'}
  Binder 3: {'A52': 'CB', 'A53': 'CB', 'A54': 'CB', 'A55': 'CA', 'A56': 'CB'}
  Binder 4: {'A71': 'CB', 'A72': 'CB', 'A73': 'CB', 'A74': 'CB', 'A75': 'CB'}
  Binder 5: {'A88': 'CB', 'A89': 'CB', 'A91': 'CB', 'A92': 'CB', 'A94': 'CB'}
  Binder 6: {'A113': 'CB', 'A114': 'CB', 'A115': 'CB', 'A116': 'CB', 'A117': 'CB'}
  Binder 7: {'A134': 'CB', 'A135': 'CB', 'A137': 'CB', 'A138': 'CB', 'A139': 'CB'}
  Binder 8: {'A152': 'CB', 'A153': 'CB', 'A154': 'CB', 'A155': 'CB', 'A156': 'CB'}
  Binder 9: {'A170': 'CB', 'A171': 'CB', 'A172': 'C

INFO:foundry.inference_engines.base:[rank: 0] Loading checkpoint from /root/.foundry/checkpoints/rfd3_latest.ckpt...
INFO:foundry.inference_engines.base:[rank: 0] Building Transform pipeline...
INFO:foundry.inference_engines.base:[rank: 0] Using settings from validation dataset: unconditional.


INFERENCE TRANSFORM PIPELINE
├── _target_
│   └── rfd3.transforms.pipelines.build_atom14_base_pipeline                                        
├── is_inference
│   └── True                                                                                        
├── return_atom_array
│   └── True                                                                                        
├── diffusion_batch_size
│   └── 1                                                                                           
├── sigma_data
│   └── 16                                                                                          
├── central_atom
│   └── CB                                                                                          
├── n_atoms_per_token
│   └── 14                                                                                          
├── association_scheme
│   └── dense                                                                                       
├── center_option
│   └── diffuse                                                                                     
├── generate_conformers
│   └── True                                                                                        
├── generate_conformers_for_non_protein_only
│   └── True                                                                                        
├── provide_reference_conformer_when_unmasked
│   └── False                                                                                       
├── ground_truth_conformer_policy
│   └── IGNORE                                                                                      
├── provide_elements_for_unindexed_components
│   └── True                                                                                        
├── use_element_for_atom_names_of_atomized_tokens
│   └── True                                                                                        
├── residue_cache_dir
│   └── /net/tukwila/ncorley/datahub/MACE-OFF23_medium                                              
├── max_binder_length
│   └── 170                                                                                         
├── atom_1d_features
│   └── ref_atom_name_chars: 256                                                                    
│       ref_element: 128                                                                            
│       ref_charge: 1                                                                               
│       ref_mask: 1                                                                                 
│       ref_is_motif_atom_with_fixed_coord: 1                                                       
│       ref_is_motif_atom_unindexed: 1                                                              
│       has_zero_occupancy: 1                                                                       
│       ref_pos: 3                                                                                  
│       ref_atomwise_rasa: 3                                                                        
│       active_donor: 1                                                                             
│       active_acceptor: 1                                                                          
│       is_atom_level_hotspot: 1                                                                    
│                                                                                                   
└── token_1d_features
    └── ref_motif_token_type: 3                                                                     
        restype: 32                                                                                 
        ref_plddt: 1                                                                                
        is_non_loopy: 1                                                                             
                                 

INFO:foundry.inference_engines.base:[rank: 0] Instantiating trainer...


INFERENCE TRAINER CONFIGURATION
├── strategy
│   └── ddp                                                                                         
├── accelerator
│   └── gpu                                                                                         
├── devices_per_node
│   └── 1                                                                                           
├── num_nodes
│   └── 1                                                                                           
├── loss
│   └── None                                                                                        
├── metrics
│   └── general_metrics:                                                                            
│         _target_: rfd3.metrics.design_metrics.AtomArrayMetrics                                    
│         compute_for_diffused_region_only: true                                                    
│         compute_ss_adherence_if_possible: false                                                   
│       backbone_metrics:                                                                           
│         _target_: rfd3.metrics.design_metrics.BackboneMetrics                                     
│       sidechain_metrics:                                                                          
│         _target_: rfd3.metrics.sidechain_metrics.SidechainMetrics                                 
│         central_atom: CB                                                                          
│         dist_threshold_min: 1.0                                                                   
│         dist_threshold_max: 2.0                                                                   
│         already_removed_virtual_atoms: false                                                      
│       metadata_metrics:                                                                           
│         _target_: rfd3.metrics.design_metrics.MetadataMetrics                                     
│       hbond_metrics:                                                                              
│         _target_: rfd3.metrics.hbonds_hbplus_metrics.HbondMetrics                                 
│         cutoff_HA_dist: 3                                                                         
│         cutoff_DA_distance: 3.5                                                                   
│                                                                                                   
├── _target_
│   └── rfd3.trainer.rfd3.AADesignTrainer                                                           
├── output_full_json
│   └── True                                                                                        
├── allow_sequence_outputs
│   └── True                                                                                        
├── cleanup_guideposts
│   └── True                                                                                        
├── cleanup_virtual_atoms
│   └── True                                                                                        
├── read_sequence_from_sequence_head
│   └── True                                                                                        
├── compute_non_clash_metrics_for_diffused_region_only
│   └── False                                                                                       
├── association_scheme
│   └── dense                                                                                       
├── n_examples_per_epoch
│   └── 2400                                                                                        
├── checkpoint_every_n_epochs
│   └── 10                                                                                          
├── validate_every_n_epochs
│   └── 4                                                                                           
├── max_epochs
│   └── 100000           

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:foundry.metrics.metric:[rank: 0] Adding metric general_metrics to the validation metrics...
INFO:foundry.metrics.metric:[rank: 0] Adding metric backbone_metrics to the validation metrics...
INFO:foundry.metrics.metric:[rank: 0] Adding metric sidechain_metrics to the validation metrics...
INFO:foundry.metrics.metric:[rank: 0] Adding metric metadata_metrics to the validation metrics...
INFO:foundry.metrics.metric:[rank: 0] Adding metric hbond_metrics to the validation metrics...
INFO:foundry.inference_engines.base:[rank: 0] Setting up model...
INFO:foundry.trainers.fabric:[rank: 0] Instantiating model...
INFO:foundry.trainers.fabric:[rank: 0] Wrapping model with EMA...
INFO:foundry.inference_engines.base:[rank: 0] Loading model weights from checkpoint...
INFO:foundry.trainers.fabric:[rank: 0] Using pre-loaded checkpoint...
INFO:foundry.traine

[1] Chain renaming: {'B': 'A', 'A': 'B'}
[1/10] Saved complex CIF: /content/binder_design_out/complex_00001.cif
[1/10] Saved complex PDB: /content/binder_design_out/complex_generated_00001.pdb
[2] Chain renaming: {'B': 'A', 'A': 'B'}
[2/10] Saved complex CIF: /content/binder_design_out/complex_00002.cif
[2/10] Saved complex PDB: /content/binder_design_out/complex_generated_00002.pdb
[3] Chain renaming: {'B': 'A', 'A': 'B'}
[3/10] Saved complex CIF: /content/binder_design_out/complex_00003.cif
[3/10] Saved complex PDB: /content/binder_design_out/complex_generated_00003.pdb
[4] Chain renaming: {'B': 'A', 'A': 'B'}
[4/10] Saved complex CIF: /content/binder_design_out/complex_00004.cif
[4/10] Saved complex PDB: /content/binder_design_out/complex_generated_00004.pdb
[5] Chain renaming: {'B': 'A', 'A': 'B'}
[5/10] Saved complex CIF: /content/binder_design_out/complex_00005.cif
[5/10] Saved complex PDB: /content/binder_design_out/complex_generated_00005.pdb
[6] Chain renaming: {'B': 'A', 'A':

## 6) Complex.pdb: re-scan → 2× ReGlyco optimization (rotamer OFF / ON)



This step uses a protein–binder complex PDB.



- If you ran the previous cell, it will have created `complex_generated.pdb` automatically.

- You can also upload your own `complex.pdb` (e.g., from another docking or design workflow).



We re-run scan to avoid relying on residue numbering equivalence.


In [ ]:
#@title Complex PDB(s): use generated binders OR upload one complex → Re-scan → optimization (rotamer OFF then ON only if needed) { display-mode: "form" }

from IPython.display import HTML

import json
from pathlib import Path


# If you did not run the RFD3 cell, set this to your own complex PDB path.
COMPLEX_PDB_PATH = 'complex_generated.pdb'  # fallback single-complex path

# True: re-scan each complex (recommended). False: reuse target scan sites (faster, but may mismatch).
USE_COMPLEX_SCAN = False


def _load_many_complex_pdbs() -> list[tuple[str, bytes]]:
    """Returns list of (filename, bytes) for all binder complexes."""

    # Preferred: binder list produced by the RFD3 cell
    if 'generated_complex_pdb_paths' in globals():
        try:
            paths = [Path(p) for p in list(generated_complex_pdb_paths) if p]
            paths = [p for p in paths if p.exists() and p.is_file()]
            if paths:
                out = []
                for p in paths:
                    out.append((p.name, p.read_bytes()))
                return out
        except Exception:
            pass

    # Fallback: user-specified single complex
    name, b = get_pdb_input(COMPLEX_PDB_PATH)
    return [(name, b)]


complex_inputs = _load_many_complex_pdbs()
print('Complexes loaded:', len(complex_inputs))


GLYCAN_ID = 'G00028MO'

opt_base = {
    'populationSize': 64,
    'maxGenerations': 10,
    'outputFormat': 'PDB',
}


def _site_key(site: str):
    try:
        n, c = site.split('_', 1)
        return (int(n), c)
    except Exception:
        return (10**9, site)


def per_site_pass(job_meta: dict, expected_sites: list[str]) -> dict[str, bool]:
    items = ((job_meta.get('results') or {}).get('results')) or []
    by_site: dict[str, list[bool]] = {}

    for it in items:
        site = it.get('residue')
        if not site:
            continue
        by_site.setdefault(site, []).append(it.get('clash_solved') is True)

    # For any expected site missing in results, treat as CLASH (no solved conformations)
    out: dict[str, bool] = {}
    for site in expected_sites:
        oks = by_site.get(site, [])
        out[site] = any(oks)
    return out


def _parse_sites_for_js(sites: list[str]) -> list[dict]:
    """Convert ['123_A', ...] -> [{'resi': 123, 'chain': 'A'}, ...]"""
    out = []
    for s in sites:
        try:
            n, c = s.split('_', 1)
            out.append({'resi': int(n), 'chain': str(c)})
        except Exception:
            continue
    # keep stable order, drop duplicates
    seen = set()
    dedup = []
    for it in out:
        k = (it['resi'], it['chain'])
        if k in seen:
            continue
        seen.add(k)
        dedup.append(it)
    return dedup


def run_opt_one(
    session_uuid: str,
    selected_glycans: dict[str, str],
    *,
    rotamer: bool,
) -> tuple[str, dict]:
    params = {**opt_base, 'enableRotamerScan': bool(rotamer)}
    job_uuid = submit_job(
        session_uuid,
        'optimization',
        selected_glycans=selected_glycans,
        parameters=params,
    )['job_uuid']
    meta = wait_job(job_uuid)
    return job_uuid, meta


binder_results = []

for binder_idx, (complex_name, complex_bytes) in enumerate(complex_inputs, start=1):
    print(f'\n=== Binder {binder_idx}/{len(complex_inputs)}: {complex_name} ===')

    complex_session = create_session_from_pdb_bytes(complex_name, complex_bytes)
    complex_session_uuid = complex_session['session_uuid']

    if USE_COMPLEX_SCAN:
        complex_scan_uuid = submit_job(complex_session_uuid, 'scan')['job_uuid']
        complex_scan_meta = wait_job(complex_scan_uuid)
        complex_sites = scan_passed_sites(complex_scan_meta)
    else:
        complex_sites = list(passed_sites)

    sites_sorted = sorted(complex_sites, key=_site_key)
    selected_complex = {site: GLYCAN_ID for site in sites_sorted}

    print('session_uuid:', complex_session_uuid)
    print('Sites:', len(sites_sorted))

    # 1) Rotamer OFF always
    job_off, meta_off = run_opt_one(complex_session_uuid, selected_complex, rotamer=False)
    url_off = job_file_url(job_off, 'output.pdb')

    pass_off = per_site_pass(meta_off, sites_sorted)
    clashed_off = [s for s in sites_sorted if not pass_off.get(s, False)]

    # 2) Rotamer ON only if OFF had any clashes
    job_on = None
    meta_on = None
    url_on = None
    clashed_on = []

    if clashed_off:
        job_on, meta_on = run_opt_one(complex_session_uuid, selected_complex, rotamer=True)
        url_on = job_file_url(job_on, 'output.pdb')

        pass_on = per_site_pass(meta_on, sites_sorted)
        clashed_on = [s for s in sites_sorted if not pass_on.get(s, False)]

    # Final conclusion per your rules
    if clashed_on:
        conclusion = 'Binder clash'
        final_url = url_on or url_off
        final_state = 'rotamer_on'
    elif clashed_off:
        conclusion = 'Sidechain issue'
        final_url = url_on or url_off
        final_state = 'rotamer_on' if url_on else 'rotamer_off'
    else:
        conclusion = 'No clashes detected'
        final_url = url_off
        final_state = 'rotamer_off'

    binder_results.append({
        'binder_idx': binder_idx,
        'complex_name': complex_name,
        'session_uuid': complex_session_uuid,
        'n_sites': len(sites_sorted),
        'sites': sites_sorted,
        'job_off': job_off,
        'url_off': url_off,
        'clashed_off': clashed_off,
        'job_on': job_on,
        'url_on': url_on,
        'clashed_on': clashed_on,
        'final_url': final_url,
        'final_state': final_state,
        'conclusion': conclusion,
    })

    print('Rotamer OFF output.pdb:', url_off)
    if url_on:
        print('Rotamer ON  output.pdb:', url_on)
    print('Conclusion:', conclusion)


# Export globals for the Mol* viewer cell
BINDER_FINAL_URLS = [r['final_url'] for r in binder_results]
BINDER_LABELS = [f"Binder {r['binder_idx']}" for r in binder_results]
BINDER_CONCLUSIONS = [r['conclusion'] for r in binder_results]
BINDER_GLYCO_SITES = [r['sites'] for r in binder_results]
BINDER_GLYCO_SITES_JS = [_parse_sites_for_js(r['sites']) for r in binder_results]


# Summary table (one row per binder)
rows = []
for r in binder_results:
    on_ran = (r['url_on'] is not None)
    rows.append(
        "<tr>"
        f"<td style='padding:8px; border-bottom:1px solid #eee; white-space:nowrap;'>{r['binder_idx']}</td>"
        f"<td style='padding:8px; border-bottom:1px solid #eee; white-space:nowrap;'>{r['conclusion']}</td>"
        f"<td style='padding:8px; border-bottom:1px solid #eee; text-align:right;'>{r['n_sites']}</td>"
        f"<td style='padding:8px; border-bottom:1px solid #eee; white-space:nowrap;'>{'yes' if on_ran else 'no'}</td>"
        f"<td style='padding:8px; border-bottom:1px solid #eee;'><a href='{r['final_url']}' target='_blank'>final output.pdb</a></td>"
        "</tr>"
    )

HTML(
    "<div style='font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; font-size:14px;'>"
    "<div style='margin:8px 0 10px 0; font-weight:700;'>Per-binder ReGlyco results</div>"
    "<table style='border-collapse:collapse; width:100%;'>"
    "<thead><tr>"
    "<th style='text-align:left; border-bottom:1px solid #ddd; padding:8px;'>Binder</th>"
    "<th style='text-align:left; border-bottom:1px solid #ddd; padding:8px;'>Conclusion</th>"
    "<th style='text-align:right; border-bottom:1px solid #ddd; padding:8px;'>Sites</th>"
    "<th style='text-align:left; border-bottom:1px solid #ddd; padding:8px;'>Rotamer ON ran?</th>"
    "<th style='text-align:left; border-bottom:1px solid #ddd; padding:8px;'>Final structure</th>"
    "</tr></thead>"
    "<tbody>"
    + "".join(rows)
    + "</tbody></table></div>"
)


Complexes loaded: 10

=== Binder 1/10: complex_generated_00001.pdb ===
session_uuid: f894964d-f420-49e1-9425-bd529053ad1c
Sites: 3
[28a3bc3e] status: leased
[28a3bc3e] status: completed
Rotamer OFF output.pdb: https://glycoshape.org/api/jobs/28a3bc3e-bdbb-4f2e-8cf5-27dd5a49929e/files/output.pdb
Conclusion: No clashes detected

=== Binder 2/10: complex_generated_00002.pdb ===
session_uuid: 08eae756-ce99-4285-bcbc-5abbf3cec07e
Sites: 3
[77299f4f] status: leased
[77299f4f] status: completed
Rotamer OFF output.pdb: https://glycoshape.org/api/jobs/77299f4f-594d-4876-a198-64e8f9630be9/files/output.pdb
Conclusion: No clashes detected

=== Binder 3/10: complex_generated_00003.pdb ===
session_uuid: df018ee5-188b-4b9c-b7b0-12728e45e8c0
Sites: 3
[e7ff43e4] status: leased
[e7ff43e4] status: completed
Rotamer OFF output.pdb: https://glycoshape.org/api/jobs/e7ff43e4-f31b-4e0e-8ea0-b6051d2d6732/files/output.pdb
Conclusion: No clashes detected

=== Binder 4/10: complex_generated_00004.pdb ===
session_

Binder,Conclusion,Sites,Rotamer ON ran?,Final structure
1,No clashes detected,3,no,final output.pdb
2,No clashes detected,3,no,final output.pdb
3,No clashes detected,3,no,final output.pdb
4,No clashes detected,3,no,final output.pdb
5,No clashes detected,3,no,final output.pdb
6,No clashes detected,3,no,final output.pdb
7,No clashes detected,3,no,final output.pdb
8,No clashes detected,3,no,final output.pdb
9,No clashes detected,3,no,final output.pdb
10,No clashes detected,3,no,final output.pdb


In [ ]:
# @title
from IPython.display import HTML, display
import json

# Expects globals from the previous cell:
# - BINDER_FINAL_URLS
# - BINDER_LABELS
# - BINDER_CONCLUSIONS
# - BINDER_GLYCO_SITES_JS  (list of [{'resi': int, 'chain': str}, ...] per binder)

urls = list(BINDER_FINAL_URLS) if 'BINDER_FINAL_URLS' in globals() else []
labels = list(BINDER_LABELS) if 'BINDER_LABELS' in globals() else []
conclusions = list(BINDER_CONCLUSIONS) if 'BINDER_CONCLUSIONS' in globals() else []
glyco_sites = list(BINDER_GLYCO_SITES_JS) if 'BINDER_GLYCO_SITES_JS' in globals() else []

if not urls:
    display(HTML("<div style='font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; padding:10px;'>No binder results found. Run the Step 6 cell first.</div>"))
else:
    html = """
<style>
  .nav-wrap {
    font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial;
    font-size: 14px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 12px;
    margin: 8px 0;
  }
  .nav-left { display:flex; flex-direction:column; gap:2px; }
  .nav-title { font-weight: 700; }
  .nav-sub { color: #666; font-size: 12px; }
  .nav-btn {
    appearance: none;
    border: 1px solid #ddd;
    background: #fafafa;
    border-radius: 10px;
    padding: 8px 12px;
    font-weight: 600;
    cursor: pointer;
  }
  .nav-btn:hover { background: #f2f2f2; }
  #molstar { width:100%; height: 640px; position: relative; border: 1px solid #eee; border-radius: 12px; overflow: hidden; }
  .mol-loading {
    position: absolute;
    inset: 0;
    display:flex;
    align-items:center;
    justify-content:center;
    background: white;
    color: #666;
    font-size: 12px;
    letter-spacing: 0.08em;
    z-index: 5;
  }
</style>

<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.css" />
<script src="https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.js"></script>

<div class="nav-wrap">
  <div class="nav-left">
    <div class="nav-title" id="binder-title"></div>
    <div class="nav-sub" id="binder-sub"></div>
  </div>
  <button class="nav-btn" id="next-binder">Next binder</button>
</div>

<div id="molstar">
  <div id="loading" class="mol-loading">LOADING&#x2026;</div>
</div>

<script>
const binderUrls = __URLS__;
const binderLabels = __LABELS__;
const binderConclusions = __CONCLUSIONS__;
const binderGlycoSites = __GLYCOSITES__;

let idx = 0;
let viewer = null;

function setHeader(i) {
  const title = document.getElementById('binder-title');
  const sub = document.getElementById('binder-sub');
  const n = binderUrls.length;
  const label = (binderLabels && binderLabels[i]) ? binderLabels[i] : `Binder ${i+1}`;
  const concl = (binderConclusions && binderConclusions[i]) ? binderConclusions[i] : '';
  const sites = (binderGlycoSites && binderGlycoSites[i]) ? binderGlycoSites[i] : [];
  const siteCount = sites.length;

  title.textContent = `${label} (${i+1}/${n})`;

  const pieces = [];
  if (concl) pieces.push(`Conclusion: ${concl}`);
  pieces.push(`Glyco sites: ${siteCount}`);
  sub.textContent = pieces.join(' \u00b7 ');
}

async function ensureViewer() {
  if (viewer) return viewer;
  viewer = await molstar.Viewer.create('molstar', {
    layoutIsExpanded: false,
    layoutShowControls: false,
    layoutShowRemoteState: false,
    layoutShowSequence: true,
    layoutShowLog: false,
  });
  return viewer;
}

async function loadBinder(i) {
  const loading = document.getElementById('loading');
  if (loading) loading.style.display = 'flex';

  setHeader(i);

  const v = await ensureViewer();
  const plugin = v.plugin;

  try {
    if (plugin && plugin.clear) await plugin.clear();
  } catch (e) {
    // ignore
  }

  const url = binderUrls[i];

  const HIGHLIGHT_COLOR = 0xFFD23F;

  try {
    const data = await plugin.builders.data.download({ url });
    const trajectory = await plugin.builders.structure.parseTrajectory(data, 'pdb');
    const model = await plugin.builders.structure.createModel(trajectory);
    const structure = await plugin.builders.structure.createStructure(model);

    // Render everything using Mol* defaults first (includes ligands/carbohydrates).
    let presetApplied = false;
    try {
      if (plugin.builders?.structure?.representation?.applyPreset) {
        await plugin.builders.structure.representation.applyPreset(structure, 'auto');
        presetApplied = true;
      }
    } catch (e) {
      // ignore
    }
    if (!presetApplied) {
      try {
        if (plugin.builders?.structure?.representation?.applyPreset) {
          await plugin.builders.structure.representation.applyPreset(structure, 'default');
          presetApplied = true;
        }
      } catch (e) {
        // ignore
      }
    }

    // Fallback: explicit polymer + ligand representations.
    if (!presetApplied) {
      const polymer = await plugin.builders.structure.tryCreateComponentStatic(structure, 'polymer');
      if (polymer) {
        await plugin.builders.structure.representation.addRepresentation(polymer, {
          type: 'cartoon',
          color: 'chain-id'
        });
      }

      const ligands = await plugin.builders.structure.tryCreateComponentStatic(structure, 'ligand');
      if (ligands) {
        await plugin.builders.structure.representation.addRepresentation(ligands, {
          type: 'ball-and-stick',
          color: 'element-symbol',
        });
      }
    }

    // Helper: add a highlighted representation for an expression component.
    async function addHighlightComponent(expr, name, sizeValue) {
      const comp = await plugin.builders.structure.tryCreateComponentFromExpression(structure, expr, name);
      if (!comp) return;
      await plugin.builders.structure.representation.addRepresentation(comp, {
        type: 'ball-and-stick',
        color: 'uniform',
        colorParams: { value: HIGHLIGHT_COLOR },
        size: 'uniform',
        sizeParams: { value: sizeValue },
      });
    }

    // Highlight glycan residues (carbohydrates) on top of defaults.
    // If residue-name properties are unavailable, fallback to highlighting all ligands.
    try {
      const MS = molstar.MolScriptBuilder;

      const glycanResnames = [
        'NAG', 'NDG', 'MAN', 'BMA', 'GLC', 'BGC', 'GAL', 'GIV',
        'FUC', 'FUL', 'SIA', 'SLB', 'NE5', 'NGC'
      ];

      function makeResnameTest(propFn) {
        const tests = glycanResnames.map(rn => MS.core.rel.eq([propFn(), rn]));
        if (tests.length === 1) return tests[0];
        return MS.core.logic.or(tests);
      }

      const hasLabelCompId = !!(MS.struct?.atomProperty?.macromolecular?.label_comp_id);
      const hasAuthCompId = !!(MS.struct?.atomProperty?.macromolecular?.auth_comp_id);

      if (hasLabelCompId) {
        const expr = MS.struct.generator.atomGroups({
          'residue-test': makeResnameTest(MS.struct.atomProperty.macromolecular.label_comp_id)
        });
        await addHighlightComponent(expr, 'glycan-label-comp', 0.35);
      } else if (hasAuthCompId) {
        const expr = MS.struct.generator.atomGroups({
          'residue-test': makeResnameTest(MS.struct.atomProperty.macromolecular.auth_comp_id)
        });
        await addHighlightComponent(expr, 'glycan-auth-comp', 0.35);
      } else {
        const ligands = await plugin.builders.structure.tryCreateComponentStatic(structure, 'ligand');
        if (ligands) {
          await plugin.builders.structure.representation.addRepresentation(ligands, {
            type: 'ball-and-stick',
            color: 'uniform',
            colorParams: { value: HIGHLIGHT_COLOR },
            size: 'uniform',
            sizeParams: { value: 0.35 },
          });
        }
      }
    } catch (e) {
      console.warn('Glycan highlighting failed:', e);
    }

    // Highlight glycosylation sites (protein residues) on top of defaults.
    const sites = (binderGlycoSites && binderGlycoSites[i]) ? binderGlycoSites[i] : [];
    if (sites.length) {
      try {
        const MS = molstar.MolScriptBuilder;

        const hasAuthIds = !!(MS.struct?.atomProperty?.macromolecular?.auth_asym_id) &&
                          !!(MS.struct?.atomProperty?.macromolecular?.auth_seq_id);
        const hasLabelIds = !!(MS.struct?.atomProperty?.macromolecular?.label_asym_id) &&
                           !!(MS.struct?.atomProperty?.macromolecular?.label_seq_id);

        for (const s of sites) {
          const chain = String(s.chain || '').trim();
          const resi = Number(s.resi);
          if (!chain || !Number.isFinite(resi)) continue;

          if (hasAuthIds) {
            const exprAuth = MS.struct.generator.atomGroups({
              'chain-test': MS.core.rel.eq([MS.struct.atomProperty.macromolecular.auth_asym_id(), chain]),
              'residue-test': MS.core.rel.eq([MS.struct.atomProperty.macromolecular.auth_seq_id(), resi]),
            });
            await addHighlightComponent(exprAuth, `glyco-auth-${chain}-${resi}`, 0.30);
          }

          if (hasLabelIds) {
            const exprLabel = MS.struct.generator.atomGroups({
              'chain-test': MS.core.rel.eq([MS.struct.atomProperty.macromolecular.label_asym_id(), chain]),
              'residue-test': MS.core.rel.eq([MS.struct.atomProperty.macromolecular.label_seq_id(), resi]),
            });
            await addHighlightComponent(exprLabel, `glyco-label-${chain}-${resi}`, 0.30);
          }
        }
      } catch (e) {
        console.warn('Glyco-site highlighting failed:', e);
      }
    }

  } catch (e) {
    console.error('Mol* load failed; falling back to loadStructureFromUrl:', e);
    try {
      await v.loadStructureFromUrl(url, 'pdb');
    } catch (e2) {
      console.error('Fallback load also failed:', e2);
    }
  }

  if (loading) loading.style.display = 'none';
}

document.getElementById('next-binder').addEventListener('click', () => {
  idx = (idx + 1) % binderUrls.length;
  loadBinder(idx);
});

// Wait for molstar to be ready before loading
function waitForMolstar(cb, tries=0) {
  if (typeof molstar !== 'undefined' && molstar.Viewer) {
    cb();
  } else if (tries < 50) {
    setTimeout(() => waitForMolstar(cb, tries + 1), 200);
  } else {
    console.error('Mol* failed to load after timeout.');
    const loading = document.getElementById('loading');
    if (loading) loading.textContent = 'ERROR: Mol* viewer failed to load.';
  }
}

waitForMolstar(() => loadBinder(idx));
</script>
"""

    html = html.replace('__URLS__', json.dumps(urls))
    html = html.replace('__LABELS__', json.dumps(labels))
    html = html.replace('__CONCLUSIONS__', json.dumps(conclusions))
    html = html.replace('__GLYCOSITES__', json.dumps(glyco_sites))

    display(HTML(html))
